# 02 — Reconstruct portfolio returns

Pull historical returns for every tradeable ticker via `YahooFinanceProvider`,
then reconstruct each portfolio's daily return series using *current weights
× historical underlying-asset returns* (design D2).

**Caveat (forward-looking, not realised):** the reconstructed series is what
*today's* book *would have* returned over history — it's not a real track
record. Stashaway statements don't expose a NAV history we could reconcile
against, so this notebook is a sanity check that returns reconstruction works,
not a backtest.


In [1]:
from datetime import date
from pathlib import Path
import warnings

import pandas as pd
import plotly.graph_objects as go

from hailmary.allocation.book_config import ROLES
from hailmary.allocation.portfolios import Role, from_parsed
from hailmary.allocation.returns import portfolio_returns
from hailmary.allocation.statements import parse_statement
from hailmary.data.providers import YahooFinanceProvider
from hailmary.viz.theme import apply_theme

STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
START = date(2022, 1, 1)
END = date.today()

## Parse + resolve

In [2]:
parsed = parse_statement(STATEMENT_PATH, use_cache=False)
portfolios = [
    from_parsed(pf, roles=ROLES[pf.name])
    for pf in parsed if pf.name in ROLES
]
holding = [p for p in portfolios if Role.HOLDING in p.roles]
print(f'{len(portfolios)} portfolios resolved, {len(holding)} tagged HOLDING')

15 portfolios resolved, 12 tagged HOLDING


## Fetch returns for the diagnostic universe

Pull every Yahoo-resolvable ticker across HOLDING portfolios in one call so
the cache key is shared across portfolios.

In [3]:
provider = YahooFinanceProvider()
tickers = sorted({
    h.metadata.ticker for p in holding for h in p.holdings
    if not h.metadata.ticker.startswith('CASH_')
})
print(f'Fetching {len(tickers)} unique tickers from Yahoo ({START} → {END})')
returns = provider.get_returns(tickers, START, END)
print(f'Returns panel: {returns.shape[0]} dates × {returns.shape[1]} tickers')
missing = sorted(set(tickers) - set(returns.columns))
if missing:
    print(f'\nTickers with NO Yahoo data: {missing}')
    print('These holdings will be excluded from return reconstruction; consider mapping to a different proxy in universe.py.')

2026-05-11 01:07:34.832 | INFO     | hailmary.data.providers.yahoo:get_bars:64 - Fetching 48 symbols from Yahoo (2022-01-01 → 2026-05-11); inclusive


Fetching 48 unique tickers from Yahoo (2022-01-01 → 2026-05-11)


2026-05-11 01:07:37.224 | DEBUG    | hailmary.data.cache:set:45 - Cached 52844 rows key=2b98582336e0


Returns panel: 1590 dates × 48 tickers


## Reconstruct one return series per portfolio

In [4]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    series = []
    failed = []
    for p in holding:
        try:
            s = portfolio_returns(p, returns=returns)
            series.append(s)
        except Exception as exc:
            failed.append((p.name, str(exc)))

panel = pd.concat(series, axis=1) if series else pd.DataFrame()
print(f'Reconstructed {len(series)} portfolio return series across {panel.shape[0]} dates')
if failed:
    print('\nFailed:')
    for n, e in failed:
        print(f'  {n}: {e}')

Reconstructed 12 portfolio return series across 1590 dates


C:\Users\Dalva\AppData\Local\Temp\ipykernel_46400\786161185.py:13: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  panel = pd.concat(series, axis=1) if series else pd.DataFrame()


## Summary stats

In [5]:
from hailmary.analytics.metrics import PerformanceMetrics
stats = []
common = panel.dropna(how='any')
for col in common.columns:
    m = PerformanceMetrics(common[col])
    stats.append({
        'portfolio': col,
        'ann_return': m.annualised_return,
        'ann_vol': m.annualised_vol,
        'sharpe': m.sharpe,
        'max_dd': m.max_drawdown,
    })
stats_df = pd.DataFrame(stats).set_index('portfolio').sort_values('sharpe', ascending=False)
stats_df.style.format({
    'ann_return': '{:.2%}', 'ann_vol': '{:.2%}', 'sharpe': '{:.2f}', 'max_dd': '{:.2%}'
})

,ann_return,ann_vol,sharpe,max_dd
portfolio,,,,
SG ETF,40.12%,11.69%,2.94,-7.31%
Singapore Investing,11.01%,3.76%,2.80,-2.50%
Utilities,30.95%,15.77%,1.79,-10.61%
Income Investing,7.05%,4.83%,1.44,-3.55%
BlackRock,15.29%,10.69%,1.38,-10.92%
General SRS,15.62%,14.51%,1.07,-12.66%
High Dividend Yield,14.23%,13.71%,1.04,-15.27%
General Investing,15.07%,14.81%,1.02,-13.33%
Ex-US Large-cap,16.39%,16.22%,1.02,-14.09%


## Cumulative-return chart

In [6]:
cum = (1 + common).cumprod() - 1
fig = go.Figure()
for col in cum.columns:
    fig.add_trace(go.Scatter(x=cum.index, y=cum[col], mode='lines', name=col))
fig.update_layout(yaxis_tickformat='.0%')
apply_theme(fig, title='Reconstructed cumulative returns', height=520)

## Tracking-error placeholder

Stashaway statements don't expose a NAV history per portfolio, so we can't
compute a real tracking error against the parser's reconstruction. If a NAV
feed becomes available later, plot `(reconstructed - actual)` here.